# ML Challenge 2026 — finish: v9 submission, embedding gate, second-stage selector (CPU)

Inputs (Add Input): the competition data, the v9 main run's output (as a dataset or notebook
output: it must contain work/lgbm.txt), and the embedding run's output (ann_train.parquet).

1. v9 submission -> /kaggle/working/submission_v9/: candidate_pairs.tsv rebuilt from the 24
   shard files (the run's own copy was cut off when its kernel died) + the run's complete
   matching_results.tsv. Validated.
2. Embedding gate: DuckDB candidates vs embedding candidates vs their union, on the 5% sample.
3. Second-stage selector -> /kaggle/working/output/: rebuilds held-out probabilities, compares
   selection rules with 5-fold CV, applies the winner to test. Validated.

In [ ]:
import glob, os, shutil, subprocess, sys, time
from pathlib import Path

import duckdb, lightgbm as lgb, numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq

DATA_DIR = Path("/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset")
if not DATA_DIR.exists():
    DATA_DIR = Path("/kaggle/input/amazon-ml-challenge-2026/dataset")
def _find(name, root="/kaggle/input"):
    for dirpath, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(dirpath)
    return None

WORK = _find("lgbm.txt")                           # wherever Kaggle mounted the main run's output
assert WORK is not None, "lgbm.txt not found under /kaggle/input: attach the main run's output"
RUN_DIR, PREV_OUT = WORK.parent, WORK.parent / "output"
OUT = Path("/kaggle/working/output")                 # selector submission
SUB_V9 = Path("/kaggle/working/submission_v9")       # plain v9 submission
for d in (OUT, SUB_V9):
    d.mkdir(parents=True, exist_ok=True)
ANN_DIR = _find("ann_train.parquet")
GT_PATH = DATA_DIR / "train" / "train_ground_truth.tsv"

def duplicate_records(folder):
    """Records given to more than one S1. The truth never does this, so each one is a wrong match."""
    seen, dup = set(), 0
    with open(folder / "matching_results.tsv") as fh:
        next(fh)
        for line in fh:
            for c in line.rstrip("\n").split("\t")[1].split(","):
                if c:
                    dup += c in seen
                    seen.add(c)
    return dup

def validate(folder):
    r = subprocess.run([sys.executable, str(DATA_DIR.parent / "validate_submission.py"),
                        "--matching", str(folder / "matching_results.tsv"),
                        "--candidate", str(folder / "candidate_pairs.tsv"),
                        "--test-dir", str(DATA_DIR / "test")], capture_output=True, text=True)
    print(r.stdout[-2500:] or r.stderr[-2000:])
    print(f"{folder.name}: exit code {r.returncode}  ->  {'PASS' if r.returncode == 0 else 'FAIL'}")
    return r.returncode == 0
PROB_FLOOR = 0.01                                   # same floor predict_shard applied to test
SOURCE_COLUMNS = ["entity_id", "business_name", "business_address", "country"]
print(f"main run: {RUN_DIR}")
print(f"embeddings: {ANN_DIR if ANN_DIR else 'not attached (the gate will be skipped)'}")
print(f"  test prob shards {len(list(WORK.glob('test_probs*.parquet')))}  "
      f"sum shards {len(list(WORK.glob('test_sums*.parquet')))}")

## 1. v9 submission (candidate_pairs.tsv rebuilt)
Built from the per-shard candidate files, one shard at a time: each shard holds whole S1 entities, so no step needs all 155M ids at once.
Runs first, before anything large is in memory.

In [ ]:
def build_candidate_pairs(out_path):
    tmp = Path("/kaggle/working/tmp_cand")
    tmp.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect()
    con.execute(f"SET memory_limit='12GB'; SET temp_directory='{tmp}'; SET preserve_insertion_order=false")
    aggs = []
    for f in sorted(WORK.glob("test_candidates*.parquet")):
        a = tmp / f"agg_{f.stem}.parquet"
        con.execute(f"""COPY (SELECT s1_id, string_agg(cand_id, ',' ORDER BY cand_id) AS c
                             FROM read_parquet('{f}') GROUP BY s1_id) TO '{a}' (FORMAT PARQUET)""")
        aggs.append(str(a))
    ids = pd.read_csv(DATA_DIR / "test" / "test_source1.tsv", sep="\t", header=0, names=SOURCE_COLUMNS,
                      usecols=["entity_id"], dtype=str)["entity_id"]
    con.register("s1ord", pa.table({"entity_id": pa.array(ids.tolist(), pa.string()),
                                    "i": pa.array(np.arange(len(ids)), pa.int64())}))
    # an entity without candidates gets NULL, which the CSV writer leaves as an empty field
    con.execute(f"""COPY (SELECT s.entity_id AS source1_entity_id, a.c AS candidate_entity_ids
                          FROM s1ord s LEFT JOIN read_parquet({aggs}) a ON a.s1_id = s.entity_id
                          ORDER BY s.i)
                    TO '{out_path}' (FORMAT CSV, DELIMITER '\t', HEADER)""")
    con.close()
    shutil.rmtree(tmp, ignore_errors=True)
    return len(aggs)

t0 = time.perf_counter()
# Always rebuilt from the shard files: a copy from a run that died mid-write would be truncated.
n = build_candidate_pairs(OUT / "candidate_pairs.tsv")
with open(OUT / "candidate_pairs.tsv") as fh:
    n_rows = sum(1 for _ in fh) - 1
print(f"candidate_pairs.tsv built from {n} shard files: {n_rows:,} rows in {time.perf_counter() - t0:.0f}s")

# v9 submission: the rebuilt candidate file + the run's own matching file (written before the crash)
shutil.copy(OUT / "candidate_pairs.tsv", SUB_V9 / "candidate_pairs.tsv")
shutil.copy(PREV_OUT / "matching_results.tsv", SUB_V9 / "matching_results.tsv")
with open(SUB_V9 / "matching_results.tsv") as fh:
    print(f"v9 matching_results.tsv: {sum(1 for _ in fh) - 1:,} rows")
V9_OK = validate(SUB_V9)
print(f"v9: records assigned to more than one S1: {duplicate_records(SUB_V9):,}")

## 2. Embedding gate: DuckDB vs embeddings vs their union (same 5% S1 sample)
Recall = share of true matches that appear among the candidates. Ceiling = the macro F0.5
a perfect model would score on those candidates. added/ent = embedding candidates per S1
that DuckDB did not already have (the extra pairs the main run would have to score).

In [ ]:
def run_gate(ks=(10, 20, 50, 100)):
    ann = str(ANN_DIR / "ann_train.parquet")
    duck = str(WORK / "train_candidates.parquet")
    g = duckdb.connect()
    g.execute(f"""
        CREATE TABLE ann AS SELECT * FROM read_parquet('{ann}');
        CREATE TABLE duck AS SELECT s1_id, cand_id FROM read_parquet('{duck}');
        CREATE TABLE ents AS SELECT DISTINCT s1_id FROM ann;
        CREATE TABLE gt AS
        SELECT s1_id, TRIM(c) AS cand_id FROM (
            SELECT source1_entity_id AS s1_id, UNNEST(STRING_SPLIT(matched_entity_ids, ',')) AS c
            FROM read_csv('{GT_PATH}', delim='\t', header=true,
                 columns={{'source1_entity_id':'VARCHAR','matched_entity_ids':'VARCHAR'}})
            WHERE matched_entity_ids <> '') JOIN ents USING (s1_id);
        CREATE TABLE m AS SELECT s1_id, COUNT(*) AS m FROM gt GROUP BY 1;""")
    n_ann = g.sql("SELECT COUNT(*) FROM ents").fetchone()[0]
    n_duck = g.sql("SELECT COUNT(DISTINCT s1_id) FROM duck").fetchone()[0]
    print(f"S1 entities: embeddings {n_ann:,}  DuckDB {n_duck:,}")
    if abs(n_ann - n_duck) > 0.01 * n_duck:
        print("different S1 samples (check s1_sample = 50 in both runs): gate skipped")
        return
    flags = ", ".join(f"MAX(CASE WHEN a.ann_rank <= {k} THEN 1 ELSE 0 END) AS a{k}" for k in ks)
    g.execute(f"""CREATE TABLE hit AS
        SELECT t.s1_id, t.cand_id, MAX(CASE WHEN d.cand_id IS NOT NULL THEN 1 ELSE 0 END) AS d, {flags}
        FROM gt t
        LEFT JOIN duck d ON d.s1_id = t.s1_id AND d.cand_id = t.cand_id
        LEFT JOIN ann a ON a.s1_id = t.s1_id AND a.cand_id = t.cand_id
        GROUP BY 1, 2""")

    def row(expr, k=None):
        r = g.sql(f"SELECT AVG({expr}) FROM hit").fetchone()[0]
        c = g.sql(f"""SELECT AVG(CASE WHEN m = 0 THEN 1.0 WHEN f = 0 THEN 0.0 ELSE 1.25 * f / (f + 0.25 * m) END)
            FROM (SELECT e.s1_id, COALESCE(m.m, 0) m, COALESCE(h.f, 0) f FROM ents e LEFT JOIN m USING (s1_id)
                  LEFT JOIN (SELECT s1_id, SUM({expr}) f FROM hit GROUP BY 1) h USING (s1_id))""").fetchone()[0]
        added = ""
        if k:
            n_new = g.sql(f"""SELECT COUNT(*) FROM ann a ANTI JOIN duck d USING (s1_id, cand_id)
                              WHERE a.ann_rank <= {k}""").fetchone()[0]
            added = f"{n_new / n_ann:10.1f}"
        return r, c, added

    base, base_c, _ = row("d")
    print(f"\n{'configuration':30s} {'recall':>7s} {'delta':>8s} {'ceiling':>8s} {'added/ent':>10s}")
    print(f"{'DuckDB A-H top 100':30s} {base:7.4f} {'':>8s} {base_c:8.4f}")
    for k in ks:
        r, c, _ = row(f"a{k}")
        print(f"{'embeddings alone top ' + str(k):30s} {r:7.4f} {'':>8s} {c:8.4f}")
    for k in ks:
        r, c, added = row(f"GREATEST(d, a{k})", k)
        print(f"{'union + embed top ' + str(k):30s} {r:7.4f} {r - base:+8.4f} {c:8.4f} {added}")

    both, d_only, a_only, neither, n = g.sql("""SELECT SUM(d*a50), SUM(d*(1-a50)), SUM((1-d)*a50),
        SUM((1-d)*(1-a50)), COUNT(*) FROM hit""").fetchone()
    print(f"\ntrue matches (embed top 50): both {both/n:.4f}  DuckDB only {d_only/n:.4f}  "
          f"embed only {a_only/n:.4f}  neither {neither/n:.4f}")
    rk = g.sql("SELECT a.ann_rank FROM hit h JOIN ann a USING (s1_id, cand_id) WHERE h.d = 0").fetchnumpy()["ann_rank"]
    if len(rk):
        print("embed-only hits by rank: " + "  ".join(f"<= {r}: {(rk <= r).mean():.0%}" for r in (5, 10, 20, 50)))

    names = [WORK / f"train_s{i}.parquet" for i in (1, 2, 3)]
    if all(p.exists() for p in names):
        print("\nexamples found only by embeddings:")
        pool = [str(names[1]), str(names[2])]
        for sn, sa, cn, ca, r, sc in g.sql(f"""
            SELECT s.name_text, s.addr_text, p.name_text, p.addr_text, a.ann_rank, a.ann_score
            FROM hit h JOIN ann a USING (s1_id, cand_id)
            JOIN read_parquet('{names[0]}') s ON s.entity_id = h.s1_id
            JOIN read_parquet({pool}) p ON p.entity_id = h.cand_id
            WHERE h.d = 0 AND a.ann_rank <= 50 ORDER BY hash(h.s1_id || h.cand_id) LIMIT 25""").fetchall():
            print(f"  rank {r:3d} cos {sc:.2f}  {(sn or '')[:34]!r:36s} vs {(cn or '')[:34]!r:36s} | "
                  f"{(sa or '')[:24]!r} vs {(ca or '')[:24]!r}")
    g.close()
    gain = a_only / n
    print(f"\nGATE: union recall gain at top 50 = {gain:+.4f} -> "
          + ("PROCEED" if gain >= 0.025 else "STOP" if gain < 0.015 else "BORDERLINE (look at the examples)"))

if ANN_DIR is None:
    print("ann_train.parquet not attached: gate skipped")
elif not (WORK / "train_candidates.parquet").exists():
    print("train_candidates.parquet not in the main run's output: gate skipped")
else:
    run_gate()

## 3. Selector: metric and the selection rules

In [ ]:
def f05(pred, true):
    k, m = len(pred), len(true)
    if k == 0 and m == 0:
        return 1.0
    if k == 0 or m == 0:
        return 0.0
    h = len(pred & true)
    return 0.0 if h == 0 else 1.25 * h / (k + 0.25 * m)

def macro_f05(pred, truth):
    return sum(f05(pred.get(s, set()), t) for s, t in truth.items()) / len(truth)

def select_base(df, e_m):
    """The main notebook's rule: cut at the k maximizing 1.25*sum(p_1..k) / (k + 0.25*E[m])."""
    d = df.sort_values(["s1_id", "p"], ascending=[True, False], kind="stable")
    p = d["p"].to_numpy(np.float64)
    g = d.groupby("s1_id", sort=False)
    k = g.cumcount().to_numpy() + 1
    em = d["s1_id"].map(e_m).fillna(0.0).to_numpy(np.float64)
    d = d.assign(score=1.25 * g["p"].cumsum().to_numpy() / (k + 0.25 * em), k=k,
                 log1mp=np.log1p(-np.clip(p, 0, 1 - 1e-12)))
    best = d.loc[d.groupby("s1_id", sort=False)["score"].idxmax(), ["s1_id", "score", "k"]].set_index("s1_id")
    score0 = np.exp(d.groupby("s1_id", sort=False)["log1mp"].sum())
    kstar = best["k"].where(~(score0 > best["score"]), 0)
    sel = d[d["k"] <= d["s1_id"].map(kstar).to_numpy()]
    out = {s: set() for s in kstar.index}
    for s, c in zip(sel["s1_id"].to_numpy(), sel["cand_id"].to_numpy()):
        out[s].add(c)
    return out

def _poisson_binomial(ps):
    dist = np.ones(1)
    for q in ps:
        dist = np.convolve(dist, [1.0 - q, q])
    return dist

def best_cut(ps, cap=30):
    """Exact expected F0.5 of predicting the top k, for every k; returns the best k.
    Pairs are treated as independent; ps must be sorted descending."""
    ps = ps[:cap]
    n = len(ps)
    suffix = [None] * (n + 1)
    suffix[n] = np.ones(1)
    for i in range(n - 1, -1, -1):
        suffix[i] = np.convolve(suffix[i + 1], [1.0 - ps[i], ps[i]])
    best_k, best = 0, suffix[0][0]                 # k = 0 scores 1 only when there is no match
    prefix = np.ones(1)
    for k in range(1, n + 1):
        prefix = np.convolve(prefix, [1.0 - ps[k - 1], ps[k - 1]])
        h = np.arange(k + 1)[:, None]
        r = np.arange(n - k + 1)[None, :]
        e = prefix @ (1.25 * h / (k + 0.25 * (h + r))) @ suffix[k]
        if e > best:
            best_k, best = k, e
    return best_k

def select_exact(df, e_m=None):
    d = df.sort_values(["s1_id", "p"], ascending=[True, False], kind="stable")
    out = {}
    sids, cids, ps = d["s1_id"].to_numpy(), d["cand_id"].to_numpy(), d["p"].to_numpy(np.float64)
    starts = np.flatnonzero(np.r_[True, sids[1:] != sids[:-1]])
    ends = np.r_[starts[1:], len(sids)]
    for a, b in zip(starts, ends):
        k = best_cut(ps[a:b])
        out[sids[a]] = set(cids[a:a + k])
    return out

def repair_exclusivity(pred, df, select, e_m, rounds=3):
    """A record can belong to one S1 only: keep it with the highest-probability claimant."""
    prob = dict(zip(zip(df["s1_id"].to_numpy(), df["cand_id"].to_numpy()), df["p"].to_numpy()))
    total = 0
    for _ in range(rounds):
        owners = {}
        for s, cs in pred.items():
            for c in cs:
                owners.setdefault(c, []).append(s)
        contested = {c: s for c, s in owners.items() if len(s) > 1}
        if not contested:
            break
        total += len(contested)
        drop = set()
        for c, ss in contested.items():
            w = max(ss, key=lambda s: prob.get((s, c), 0.0))
            drop.update((s, c) for s in ss if s != w)
        keep = ~pd.MultiIndex.from_arrays([df["s1_id"], df["cand_id"]]).isin(drop)
        df = df[keep]
        pred = select(df, e_m)
    return pred, total

# self-checks: the exact rule on hand-computable cases
assert best_cut(np.array([0.99])) == 1 and best_cut(np.array([0.05])) == 0
assert best_cut(np.array([0.95, 0.9, 0.02])) == 2
assert f05({"a", "b", "c"}, {"a", "c"}) == 1.25 * 2 / (3 + 0.5)
print("selection checks PASS")

## Held-out probabilities (calib + val folds of the main run)

In [ ]:
booster = lgb.Booster(model_file=str(WORK / "lgbm.txt"))
_cal = np.load(WORK / "calib.npz")
calibrate = lambda raw: np.interp(raw, _cal["x"], _cal["y"])   # = IsotonicRegression.predict (clip)
FEATURES = booster.feature_name()

def fold_of(s1_ids):                               # identical to the main notebook
    h = pd.util.hash_array(np.asarray(s1_ids), hash_key="0" * 16) % 100
    return np.where(h < 70, 0, np.where(h < 85, 1, 2))

t0 = time.perf_counter()
pf = pq.ParquetFile(WORK / "train_features.parquet")
parts = []
for batch in pf.iter_batches(batch_size=1_000_000, columns=["s1_id", "cand_id"] + FEATURES):
    b = batch.to_pandas()
    b = b[fold_of(b["s1_id"].to_numpy()) > 0]
    if len(b):
        p = calibrate(booster.predict(b[FEATURES]))
        parts.append(pd.DataFrame({"s1_id": b["s1_id"].to_numpy(), "cand_id": b["cand_id"].to_numpy(), "p": p}))
held = pd.concat(parts, ignore_index=True)
del parts

gt = pd.read_csv(DATA_DIR / "train" / "train_ground_truth.tsv", sep="\t", dtype=str,
                 keep_default_na=False, na_values=[])
ents = set(held["s1_id"].unique())
truth = {s: (set(m.split(",")) if m else set())
         for s, m in zip(gt["source1_entity_id"], gt["matched_entity_ids"]) if s in ents}
pos = {(s, c) for s, cs in truth.items() for c in cs}
held["y"] = [int(k in pos) for k in zip(held["s1_id"].to_numpy(), held["cand_id"].to_numpy())]

# mimic the test files: E[m] from all candidates, pairs kept only above the floor
held_em = held.groupby("s1_id")["p"].sum()
held = held[held["p"] >= PROB_FLOOR].reset_index(drop=True)
print(f"held-out entities {len(truth):,}  pairs above floor {len(held):,}  "
      f"positives {held['y'].sum():,}  ({time.perf_counter() - t0:.0f}s)")

## Second-stage features: each pair's place in its entity's probability profile

In [ ]:
S2_FEATURES = ["p", "rank", "n", "e_m", "p_max", "p_2nd", "p_rel", "gap_top", "gap_prev",
               "gap_next", "cum_p", "rest_p", "n_over_half", "logit_p"]

def s2_features(df, e_m):
    d = df.sort_values(["s1_id", "p"], ascending=[True, False], kind="stable").reset_index(drop=True)
    g = d.groupby("s1_id", sort=False)["p"]
    d["rank"] = g.cumcount() + 1
    d["n"] = g.transform("size")
    d["e_m"] = d["s1_id"].map(e_m).fillna(0.0)
    d["p_max"] = g.transform("max")
    d["p_2nd"] = d["s1_id"].map(d.loc[d["rank"] == 2].set_index("s1_id")["p"]).fillna(0.0)
    d["p_rel"] = d["p"] / d["p_max"].clip(lower=1e-9)
    d["gap_top"] = d["p_max"] - d["p"]
    d["gap_prev"] = (g.shift(1) - d["p"]).fillna(0.0)
    d["gap_next"] = (d["p"] - g.shift(-1)).fillna(d["p"])
    d["cum_p"] = g.cumsum()
    d["rest_p"] = d["e_m"] - d["p"]
    d["n_over_half"] = (d["p"] > 0.5).groupby(d["s1_id"], sort=False).transform("sum")
    d["logit_p"] = np.log(np.clip(d["p"], 1e-6, 1 - 1e-6) / np.clip(1 - d["p"], 1e-6, 1))
    return d

S2_PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=15, min_data_in_leaf=200,
                 feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                 seed=7, deterministic=True, verbosity=-1, num_threads=4)
S2_ROUNDS = 300

def rescored_em(e_m, before, after):
    """E[m] after re-scoring: the mass below the floor is kept, the rest is replaced."""
    idx = e_m.index
    return (e_m - before.groupby("s1_id")["p"].sum().reindex(idx).fillna(0.0)
            + after.groupby("s1_id")["p"].sum().reindex(idx).fillna(0.0))

def fit_s2(d):
    return lgb.train(S2_PARAMS, lgb.Dataset(d[S2_FEATURES], d["y"]), num_boost_round=S2_ROUNDS)

held2 = s2_features(held, held_em)

# must equal "expected-F_0.5 selection" in the main run's log: proves the probabilities match
val_truth = {s: t for s, t in truth.items() if fold_of(np.array([s]))[0] == 2}
print(f"val fold, current rule: {macro_f05(select_base(held, held_em), val_truth):.4f}  "
      f"<- compare with the main run's 'expected-F_0.5 selection' line")

## Compare the rules with 5-fold CV by entity

In [ ]:
all_ents = np.array(sorted(truth))
cv_fold = pd.util.hash_array(all_ents, hash_key="1" * 16) % 5
fold_of_ent = dict(zip(all_ents, cv_fold))
held2["cv"] = held2["s1_id"].map(fold_of_ent)

methods = {"base": [], "exact": [], "s2 + base": [], "s2 + exact": []}
t0 = time.perf_counter()
for f in range(5):
    tr, te = held2[held2["cv"] != f], held2[held2["cv"] == f]
    te_truth = {s: truth[s] for s in all_ents[cv_fold == f]}
    q = te.assign(p=fit_s2(tr).predict(te[S2_FEATURES]))
    q_em = rescored_em(held_em, te, q)
    methods["base"].append(macro_f05(select_base(te, held_em), te_truth))
    methods["exact"].append(macro_f05(select_exact(te), te_truth))
    methods["s2 + base"].append(macro_f05(select_base(q, q_em), te_truth))
    methods["s2 + exact"].append(macro_f05(select_exact(q), te_truth))
print(f"  5-fold CV in {time.perf_counter() - t0:.0f}s on {len(all_ents):,} held-out entities\n")

base_mean = np.mean(methods["base"])
print(f"  {'method':12s} {'macro F0.5':>10s} {'vs base':>9s}   per fold (vs base)")
for name, scores in methods.items():
    diffs = np.array(scores) - np.array(methods["base"])
    print(f"  {name:12s} {np.mean(scores):10.4f} {np.mean(scores) - base_mean:+9.4f}   "
          + " ".join(f"{x:+.4f}" for x in diffs))
# a rule wins only if it is better on average AND in at least 4 of the 5 folds
winners = {n: np.mean(s) - base_mean for n, s in methods.items()
           if n != "base" and np.mean(s) > base_mean
           and (np.array(s) > np.array(methods["base"])).sum() >= 4}
CHOSEN = max(winners, key=winners.get) if winners else "base"
print(f"\nCHOSEN: {CHOSEN}  ({np.mean(methods[CHOSEN]) - base_mean:+.4f} vs base on held-out entities)")

## Apply to the test probabilities and write the submission

In [ ]:
t0 = time.perf_counter()
probs = pd.concat([pd.read_parquet(f) for f in sorted(WORK.glob("test_probs*.parquet"))], ignore_index=True)
sums = pd.concat([pd.read_parquet(f) for f in sorted(WORK.glob("test_sums*.parquet"))],
                 ignore_index=True).groupby("s1_id")["e_m"].sum()
print(f"test pairs above floor {len(probs):,}  entities with candidates {len(sums):,}")

if CHOSEN.startswith("s2"):
    model = fit_s2(held2)                          # all held-out entities
    t2 = s2_features(probs, sums)
    test = t2[["s1_id", "cand_id"]].assign(p=model.predict(t2[S2_FEATURES]))
    em_test = rescored_em(sums, probs, test)
else:
    test, em_test = probs, sums

select = select_exact if CHOSEN.endswith("exact") else select_base
pred = select(test, em_test)
pred, contested = repair_exclusivity(pred, test, select, em_test)
print(f"  contested records resolved {contested:,}  ({time.perf_counter() - t0:.0f}s)")

s1_ids = pd.read_csv(DATA_DIR / "test" / "test_source1.tsv", sep="\t", header=0, names=SOURCE_COLUMNS,
                     usecols=["entity_id"], dtype=str)["entity_id"].tolist()
with open(OUT / "matching_results.tsv", "w") as fh:
    fh.write("source1_entity_id\tmatched_entity_ids\n")
    for s in s1_ids:
        fh.write(f"{s}\t{','.join(sorted(pred.get(s, ())))}\n")

k = np.array([len(pred.get(s, ())) for s in s1_ids])
print(f"  wrote {len(s1_ids):,} rows: empty {np.mean(k == 0):.1%}  mean set size {k.mean():.2f}")
country = pd.read_csv(DATA_DIR / "test" / "test_source1.tsv", sep="\t", header=0, names=SOURCE_COLUMNS,
                      usecols=["country"], dtype=str)["country"].to_numpy()
for c in sorted(set(country)):
    kc = k[country == c]
    print(f"    {c:7s} empty {np.mean(kc == 0):.1%}  mean set size {kc.mean():.2f}")

## Validate the selector submission (must print PASS)

In [ ]:
validate(OUT)
d_v9, d_out = duplicate_records(SUB_V9), duplicate_records(OUT)
print(f"records assigned to more than one S1:  v9 {d_v9:,}   selector output {d_out:,}")
print("\nThe v9 run's exclusivity repair restarted from the full candidate list each round, so records")
print("it had already reassigned could come back. The selector's repair keeps its removals, so")
print(f"submit {OUT} (rule: {CHOSEN}). {SUB_V9} is kept only for comparison.")